# DBSCAN Fraud Detection

This notebook implements DBSCAN clustering for credit card fraud detection using PySpark.

In [ ]:
# Import statements with issues
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler as SklearnScaler
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import time
import logging

# Problematic imports that will cause ModuleNotFoundError
from dbscan import create_spark_session
from dbscan import load_credit_card_data, preprocess_data, split_data

In [ ]:
# Spark session creation function (should be available in notebook)
def create_spark_session(app_name="DBSCAN_Fraud_Detection"):
    """
    Create and configure Spark session for fraud detection
    """
    spark = SparkSession.builder \
        .appName(app_name) \
        .config("spark.sql.adaptive.enabled", "true") \
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
        .getOrCreate()
    
    spark.sparkContext.setLogLevel("WARN")
    return spark

In [ ]:
# Data loading function (should be available in notebook)
def load_credit_card_data(spark, file_path="creditcard.csv"):
    """
    Load credit card fraud dataset
    """
    df = spark.read.option("header", "true").option("inferSchema", "true").csv(file_path)
    return df

In [ ]:
# Data preprocessing function (should be available in notebook)
def preprocess_data(df):
    """
    Preprocess the credit card data for DBSCAN clustering
    """
    # Select features for clustering (excluding Time and Class)
    feature_cols = [col for col in df.columns if col not in ['Time', 'Class']]
    
    # Assemble features
    assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
    df_assembled = assembler.transform(df)
    
    # Scale features
    scaler = StandardScaler(inputCol="features", outputCol="scaled_features")
    scaler_model = scaler.fit(df_assembled)
    df_scaled = scaler_model.transform(df_assembled)
    
    return df_scaled, feature_cols

In [ ]:
# Data splitting function (should be available in notebook)
def split_data(df, test_size=0.2, random_seed=42):
    """
    Split data into training and testing sets
    """
    train_df, test_df = df.randomSplit([1.0 - test_size, test_size], seed=random_seed)
    return train_df, test_df

In [ ]:
class ImprovedDBSCANFraudDetector:
    """
    Improved DBSCAN-based fraud detection system
    """
    
    def __init__(self, eps=0.5, min_samples=5):
        self.eps = eps
        self.min_samples = min_samples
        self.dbscan = None
        self.scaler = None
        self.feature_cols = None
        
        # This will cause import error
        from dbscan import create_spark_session
        self.spark = create_spark_session()
    
    def fit(self, X, feature_cols=None):
        """
        Fit DBSCAN model on training data
        """
        self.feature_cols = feature_cols
        
        # Scale the data
        self.scaler = SklearnScaler()
        X_scaled = self.scaler.fit_transform(X)
        
        # Fit DBSCAN
        self.dbscan = DBSCAN(eps=self.eps, min_samples=self.min_samples)
        self.dbscan.fit(X_scaled)
        
        return self
    
    def predict(self, X):
        """
        Predict anomalies using fitted DBSCAN model
        """
        if self.dbscan is None or self.scaler is None:
            raise ValueError("Model must be fitted before prediction")
        
        X_scaled = self.scaler.transform(X)
        clusters = self.dbscan.fit_predict(X_scaled)
        
        # Mark outliers (cluster -1) as fraud
        predictions = (clusters == -1).astype(int)
        return predictions
    
    def evaluate_performance(self, X_test, y_test):
        """
        Evaluate model performance
        """
        predictions = self.predict(X_test)
        
        print("Classification Report:")
        print(classification_report(y_test, predictions))
        
        print("\nConfusion Matrix:")
        print(confusion_matrix(y_test, predictions))
        
        return predictions

In [ ]:
# Main execution with problematic imports
def main():
    # This will cause import errors
    spark = create_spark_session()  # This function call will fail due to import error
    
    # Load and preprocess data
    df = load_credit_card_data(spark)  # This will also fail
    df_processed, feature_cols = preprocess_data(df)  # This will also fail
    
    # Split data
    train_df, test_df = split_data(df_processed)  # This will also fail
    
    # Convert to pandas for sklearn
    train_pandas = train_df.toPandas()
    test_pandas = test_df.toPandas()
    
    # Extract features and labels
    X_train = train_pandas[feature_cols]
    y_train = train_pandas['Class']
    X_test = test_pandas[feature_cols]
    y_test = test_pandas['Class']
    
    # Initialize and train detector
    detector = ImprovedDBSCANFraudDetector(eps=0.3, min_samples=10)
    detector.fit(X_train, feature_cols)
    
    # Evaluate performance
    predictions = detector.evaluate_performance(X_test, y_test)
    
    # Note: test_scalability function is missing and would be called here
    # test_scalability(detector, X_train, y_train)  # This function doesn't exist
    
    spark.stop()

if __name__ == "__main__":
    main()